# Plan de proyecto del chatbot Fixify: Bot, Backend y Frontend

Plan basado en los tres PDF originales, leídos el 4 de octubre de 2026: Product_Backlog_y_Requisitos_Sistema_Mantenimiento.pdf (§5.5–5.7 y reglas de negocio), Requerimientos del sistema.pdf (§4.5–4.7, §5.5–5.7 y §7), IEEE830_Fixify.pdf (§1.2, §2.1, §3.2 y §3.4). Los documentos son fuentes de requisitos; este notebook propone cómo implementarlos. Las tareas y contratos son propuestas técnicas, no requisitos textuales adicionales.



## 1. Objetivo y límites

Entregar un chatbot con Meta que consulte reparaciones y preguntas frecuentes mediante Backend, acompañado por las funciones necesarias de administración y notificaciones en el Frontend existente. El sistema conservará una única lógica de negocio en Backend y PostgreSQL central.

Esta etapa incluye HU-16–HU-22, RF-17–RF-19 y RN-08–RN-11. HU-22 tiene prioridad baja en el backlog, pero se incluye en el plan de alcance solicitado; la prioridad define el orden, no se usa para darla por implementada.

No se incluye editor de flujos, creación de comportamientos nuevos, edición arbitraria de disparadores, IA generativa ni administración de código. Las palabras que activan los flujos pueden permanecer en TypeScript. El administrador actualiza contenido FAQ y periodos de recordatorio; esto no exige editar la estructura del diálogo.



## 2. Arquitectura que cumple los documentos

```mermaid
flowchart LR
    A[Administrador] --> F[Frontend de escritorio]
    F --> B[Backend: autenticación, permisos y reglas]
    C[Cliente WhatsApp] --> M[Meta]
    M --> T[Bot BuilderBot]
    T --> B
    B --> D[(PostgreSQL central)]
    S[Proceso programado Backend] --> B
    B --> E[Solicitud de envío autorizada]
    E --> T
    T --> M
```

**Fuente:** IEEE §1.2 y §2.1; RF-19 y RN-11. El chatbot no consulta directamente PostgreSQL. Configuración, preguntas/respuestas, órdenes y registros de notificación pertenecen al Backend y su base central.

**Corrección necesaria del prototipo:** `src/database/postgres.ts` conecta directamente con PostgreSQLAdapter. Mantenerlo conectado a la base central seguiría contradiciendo el acceso mediante Backend. Para persistir eventos/contactos de BuilderBot sin romper el requisito, implementar un adaptador HTTP compatible con BuilderBot y endpoints internos de persistencia en Backend. PostgreSQL sigue siendo la base del sistema, pero sus credenciales permanecen en Backend. El historial conversacional es una propuesta de soporte técnico; no sustituye los eventos de negocio ni se presenta como un requisito funcional independiente.



## 3. Matriz de trabajo y aceptación

| Función / requisito | Bot | Backend y PostgreSQL | Frontend | Evidencia de aceptación |
| --- | --- | --- | --- | --- |
| Consulta de servicio: HU-16/17/21, RF-17/19, RN-08/11 | Capturar dato acordado; cancelar; validar; mostrar sólo resumen público; manejar ausencia y fallos | Definir identificación/verificación; comprobar propietario; devolver estado, equipo y actualización; reutilizar lógica de consulta | Integrar consulta por el mismo Backend; asegurar acceso móvil/escritorio previsto en HU-16, sin asumir que la interfaz de escritorio cubre móvil | El mismo servicio da el mismo resultado en consulta web y bot; servicio ajeno no revela datos |
| FAQ: HU-22 | Consultar catálogo Backend; responder pregunta conocida; reconocer ausencia de respuesta | Persistir preguntas y respuestas; API de lectura para bot y mantenimiento con permiso administrador | Pantalla para agregar y editar preguntas/respuestas; aplicar cambios mediante API | Un cambio administrativo se refleja en una nueva consulta sin editar TypeScript; pregunta desconocida no produce respuesta inventada |
| Aviso Listo: HU-18, RF-18, RN-09 | Enviar por Meta una solicitud autorizada y reportar resultado | Al pasar a Listo generar registro/evento; procesar envío; conservar resultado; evitar dependencia del cambio de estado respecto del canal | Actualizar estado mediante API; mostrar estado y resultado de notificación pertinente | Un cambio a Listo genera evento; Meta caído no revierte ni pierde el cambio |
| Equipo no recogido: HU-19, RN-10 | Enviar recordatorio solicitado por Backend | Guardar periodo configurable; detectar órdenes Listo vencidas; controlar duplicados | Formulario administrativo del periodo, con validación | Cambiar periodo altera elegibilidad; ejecutar proceso dos veces no duplica envíos sin control |
| Mantenimiento preventivo: HU-20, RN-10 | Enviar recordatorio solicitado por Backend | Programar desde Entregado; vincular cliente, equipo y servicio; aplicar periodo configurable | Formulario administrativo del periodo | No programa servicios sin entregar; registra referencias y fecha; no fija seis meses como constante obligatoria |
| Calidad: RNF-01–RNF-05 aplicables | Errores comprensibles; regresión; medir latencia | Permisos, hash seguro para cuentas del sistema, integridad, conservación ante fallos y respaldo/restauración | Validación de formularios, mensajes claros y permisos visibles | Evidencias de rendimiento, permisos, fallos e integridad; restauración de datos que soportan el módulo |

HU-16 menciona móvil/escritorio y HU-21 exige coincidencia con consulta web. Dado que el repositorio tiene Frontend de escritorio, la consulta web/móvil es una dependencia por resolver con el equipo: no debe declararse cubierta sólo por Tkinter.



## 4. Paquetes de trabajo en orden

### P1 — Acordar contratos y resolver arquitectura

- **Backend:** acordar formato de folio, campos públicos y mecanismo para verificar al cliente. HU-16 deja este mecanismo pendiente; teléfono verificado u OTP son opciones técnicas, no decisiones ya tomadas por los PDF.
- **Bot:** documentar entradas y salidas; reemplazar acceso PostgreSQL directo por adaptador HTTP, si se conservará historial; probar los métodos que necesita BuilderBot.
- **Frontend:** definir pantallas FAQ, periodos y consulta; precisar cómo cubrir consulta web/móvil.
- **Salida:** contratos revisables, diagrama aprobado por el equipo y criterios de prueba. No activar consulta real antes de completar propiedad y verificación.

### P2 — Consulta de reparación completa

- **Backend:** endpoint público autorizado para resumen, con identidad del bot y verificación del cliente; misma lógica que la consulta web.
- **Bot:** conectar flujo existente al endpoint real; validar folio y DTO; distinguir errores de infraestructura y consulta no verificada.
- **Frontend:** integrar consulta y datos mínimos autorizados, sin notas internas.
- **Salida:** conversación real por Meta y comprobación del mismo estado en ambos canales. El fixture DEMO-001 no satisface por sí solo RF-19.

### P3 — Evento de equipo listo

- **Backend:** persistir cambio de estado y evento de notificación de forma coherente; procesar envío separado del cambio; guardar fecha, canal, destinatario y resultado.
- **Bot:** servicio de envío Meta con acceso interno autenticado; devolver éxito o fallo al Backend.
- **Frontend:** mantener actualización de estado y presentar resultados pertinentes.
- **Salida:** orden permanece Listo incluso si el canal falla. El mecanismo de cola/outbox es una propuesta para lograrlo, no una tecnología exigida por el requisito.

### P4 — FAQ administrables

- **Backend:** modelo, validaciones y endpoints de mantenimiento; exigir administrador para cambios y exponer sólo contenido autorizado al bot.
- **Frontend:** listado y edición de preguntas/respuestas, con errores de guardado y permisos claros.
- **Bot:** flujo de consulta de FAQ, sin inventar respuestas ni abrir un editor de comportamientos.
- **Salida:** administrador cambia una respuesta y el bot usa el contenido actualizado en la siguiente consulta. Caché, si se usa, requiere actualización definida y verificable.

### P5 — Recordatorios configurables

- **Backend:** almacenar periodos; proceso programado para Listo no recogido y Entregado con mantenimiento pendiente; registrar programación y control de duplicados.
- **Frontend:** editar ambos periodos con rol administrador y unidades explícitas.
- **Bot:** reutilizar envío Meta; no implementar otra agenda independiente de la del Backend.
- **Salida:** pruebas con fechas ficticias y avance controlado del tiempo; registro de envíos sin esperar meses reales.

### P6 — Integración, calidad y entrega

- **Equipo:** pruebas E2E, permisos, fallos de Meta/Backend, rendimiento aproximado de dos segundos bajo carga normal (RNF-01), integridad y respaldo/restauración.
- **Bot:** revisar configuración Meta vigente y mecanismos necesarios para envíos proactivos; no asumir que un mensaje iniciado por el sistema se envía igual que una respuesta a consulta.
- **Salida:** notebook actualizado, manual de configuración, contratos, evidencias y defectos corregidos.



## 5. Calendario integrado

Las fechas y entregas siguen IEEE §3.2; la distribución detallada del trabajo del bot es una propuesta. Los responsables son los roles del equipo, no asignaciones individuales nuevas.

| Sprint 2026 | Bot | Backend | Frontend | Hito |
| --- | --- | --- | --- | --- |
| 4: 5–11 octubre | P1: contratos y diseño del adaptador HTTP | Identificación, privacidad y modelos necesarios | Diseño de consulta y administración | Arquitectura y contratos definidos |
| 5: 12–18 octubre | P2: integrar consulta real | Consulta autorizada y persistencia necesaria | Integrar consulta disponible y resolver canal web/móvil | Consulta integrada en entorno de pruebas |
| 6: 19–25 octubre | P3: envío Meta y errores | Evento Listo y registro de resultado | Estado y resultado del aviso | Pruebas del aviso y coherencia de consulta |
| 7: 26–30 octubre | Integración y regresión | Cerrar dependencias RF-17/18/19 | Corregir integración | Beta para entrega del 30 de octubre |
| 8: 2–8 noviembre | Robustez y envíos | P5: procesos de recordatorios y permisos | Formularios y permisos | Fallos controlados y recordatorios probados |
| 9: 9–15 noviembre | P4: FAQ y actualización de contenido | FAQ, configuración y respaldo/restauración | FAQ y periodos administrables | Versión candidata con alcance completo |
| 10: 16–22 noviembre | P6: latencia y regresión | Integridad, confiabilidad y evidencias | Usabilidad y correcciones | Evidencias de calidad |
| 11: 23–29 noviembre | Manual y demo E2E | Correcciones y despliegue | Aceptación de pantallas | Versión final para entrega del 30 de noviembre |

No se consideran completadas las entregas por el simple hecho de tener fechas en este plan. El avance se acredita con los criterios de aceptación.



## 6. Contratos propuestos entre módulos

Las rutas siguientes son propuestas de diseño; no existen todas en el repositorio ni los documentos obligan a usar estos nombres.

| Contrato | Consumidor | Permisos / responsabilidad |
| --- | --- | --- |
| Consulta de servicio y verificación | Bot y consulta web | Backend valida identidad y propiedad; retorna campos públicos |
| Lectura de FAQ | Bot | Sólo contenido autorizado del catálogo |
| Mantenimiento de FAQ | Frontend | Usuario administrador autenticado |
| Lectura/actualización de periodos | Frontend y proceso Backend | Administrador modifica; Backend aplica las reglas |
| Solicitud de envío / resultado | Backend y bot | Autenticación entre servicios; identificación del evento; control de duplicados |
| Persistencia conversacional, si se conserva | Adaptador HTTP del bot | Token de servicio restringido; Backend escribe PostgreSQL; no usar credenciales del administrador |



## 7. Flujos que se entregarán

```mermaid
flowchart TD
    A[Cliente consulta reparación] --> B[Bot captura dato acordado]
    B --> C[Backend verifica identidad y propiedad]
    C --> D{Autorizado}
    D -->|Sí| E[Resumen público]
    D -->|No| F[Respuesta genérica]
    G[Cliente pregunta FAQ] --> H[Bot consulta catálogo Backend]
    H --> I{Hay respuesta autorizada}
    I -->|Sí| J[Respuesta actualizada]
    I -->|No| K[Indicar que no hay respuesta]
```

```mermaid
sequenceDiagram
    actor Administrador
    participant Frontend
    participant Backend
    participant DB as PostgreSQL central
    participant Bot
    Administrador->>Frontend: Cambiar respuesta FAQ o periodo
    Frontend->>Backend: Petición con sesión del administrador
    Backend->>Backend: Validar rol y datos
    Backend->>DB: Guardar configuración
    Backend-->>Frontend: Confirmación
    Note over Backend,Bot: FAQ se consulta al atender pregunta; periodos los aplica Backend
    Bot->>Backend: Consultar FAQ para nueva pregunta
    Backend->>DB: Leer contenido vigente
    Backend-->>Bot: Respuesta autorizada
```



## 8. Pruebas y definición de terminado

| Prueba | Resultado esperado |
| --- | --- |
| Folio propio verificado | Estado, equipo y actualización correctos |
| Folio ajeno/inexistente | No expone existencia ni datos de terceros |
| Dato inválido/cancelación | Repite captura o termina de forma controlada |
| Cambio de FAQ por administrador | Nueva consulta usa la nueva respuesta |
| Edición FAQ/periodos por usuario sin permiso | Backend rechaza aunque se intente fuera del Frontend |
| Orden cambia a Listo con Meta caído | Orden conserva estado y queda registrado el fallo |
| Proceso de recordatorios repetido | No genera duplicados sin control |
| Servicio sin entregar | No genera recordatorio preventivo |
| Bot y consulta web | Resumen consistente de la misma orden |
| Caída Backend / reinicio del bot | Respuesta controlada y recuperación comprobada; no inventa información |
| Respaldo/restauración | Recupera configuración y registros relacionados según RNF-05 del IEEE |

Un paquete termina cuando pasan sus criterios E2E y están documentados contrato, configuración y limitaciones. Las pruebas unitarias del prototipo no acreditan por sí solas integración de Backend, Frontend o Meta.



## 9. Punto de partida y pendientes reales

Existen un flujo de consulta con demo/API, cliente HTTP con validación, configuración Meta, adaptador PostgreSQL directo y pruebas del prototipo. Los módulos actuales de Backend y Frontend se centran en autenticación, usuarios, clientes y equipos; esta revisión no encontró módulos de FAQ o recordatorios.

Pendientes: endpoint de consulta definitivo; verificación del cliente; adaptación de persistencia a Backend; catálogo FAQ; pantallas administrativas; registro/procesamiento de notificaciones; recordatorios; integración web/móvil; prueba real por Meta. Este plan no implementa esas funciones ni cambia el código de negocio.
